In [1]:
%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.displaylimit = 20

import duckdb

conn = duckdb.connect(database=":memory:")
%sql conn

%sql --file sample_data.sql

3 rows affected.

18 rows affected.

214 rows affected.

36 rows affected.

Count
36


## Data preparation

The `device_activity_clean` view removes exact duplicates and excludes activity before the device's activation date.


In [2]:
%%sql
-- Assume activation_date is correct. Exclude earlier activity and remove exact duplicates.
CREATE OR REPLACE VIEW device_activity_clean AS
SELECT DISTINCT a.*
FROM device_activity AS a
JOIN devices AS d ON a.device_id = d.device_id
WHERE a.activity_date >= d.activation_date;


Count


## 1. MAU by country


In [3]:
%%sql
SELECT
    d.country, 
    STRFTIME(da.activity_date, '%Y-%m') AS activity_month,
    COUNT(DISTINCT da.device_id) AS MAU
FROM device_activity_clean da
JOIN devices d
    ON da.device_id = d.device_id
WHERE da.activity_date >= '2026-07-01'
    AND da.activity_date < '2026-10-01'
GROUP BY 1, 2
ORDER BY 1, 2 DESC

country,activity_month,MAU
DE,2026-09,5
DE,2026-08,4
DE,2026-07,4
ES,2026-09,3
ES,2026-08,4
ES,2026-07,3
FR,2026-09,4
FR,2026-08,6
FR,2026-07,3


## 2. Month-over-month growth


In [4]:
%%sql
-- Return NULL for previous MAU and MoM growth if the previous month is missing.
WITH monthly_activity AS (
    SELECT
        d.country,
        DATE_TRUNC('month', da.activity_date) AS activity_month,
        COUNT(DISTINCT da.device_id) AS MAU
    FROM device_activity_clean da
    JOIN devices d
        ON da.device_id = d.device_id
    GROUP BY 1, 2
)

SELECT
    ma.country,
    STRFTIME(ma.activity_month, '%Y-%m') AS month,
    ma.MAU,
    pma.MAU AS prev_MAU,
    ROUND((ma.MAU - pma.MAU) * 100.0 / NULLIF(pma.MAU, 0), 2) AS MoM_change_pct
FROM monthly_activity ma
LEFT JOIN monthly_activity pma
    ON ma.country = pma.country
    AND pma.activity_month = ma.activity_month - INTERVAL '1 month'
ORDER BY 1, 2 DESC;

country,month,MAU,prev_MAU,MoM_change_pct
DE,2026-09,5,4,25.0
DE,2026-08,4,4,0.0
DE,2026-07,4,3,33.33
DE,2026-06,3,None,None
ES,2026-09,3,4,-25.0
ES,2026-08,4,3,33.33
ES,2026-07,3,None,None
FR,2026-09,4,6,-33.33
FR,2026-08,6,3,100.0
FR,2026-07,3,3,0.0


## 3. Top brands


In [5]:
%%sql
-- Break ties in minutes watched alphabetically by brand.

WITH ranked_brands AS (
    SELECT
        d.country,
        d.brand,
        SUM(da.minutes_watched) AS total_minutes_watched,
        ROW_NUMBER() OVER (PARTITION BY d.country ORDER BY SUM(da.minutes_watched) DESC, d.brand) AS rank
    FROM device_activity_clean da
    JOIN devices d
        ON da.device_id = d.device_id
    WHERE da.activity_date >= '2026-09-01'
        AND da.activity_date < '2026-10-01'
    GROUP BY 1, 2
)

SELECT
    country,
    brand,
    total_minutes_watched,
    rank
FROM ranked_brands
WHERE rank <= 2
ORDER BY 1, 4


country,brand,total_minutes_watched,rank
DE,Brand A,2705,1
DE,Brand C,480,2
ES,Brand A,1415,1
ES,Brand C,60,2
FR,Brand B,2080,1
FR,Brand A,1290,2


## 4. Cohort retention


In [6]:
%%sql
-- Assume activity data is complete through the latest activity month.

WITH activity AS (
    SELECT DISTINCT
        device_id,
        DATE_TRUNC('month', activity_date) AS activity_month
    FROM device_activity_clean
),

activation AS (
    SELECT
        device_id,
        DATE_TRUNC('month', activation_date) AS activation_month
    FROM devices
),

cohort_activity AS (
    SELECT
        act.device_id,
        act.activation_month,
        DATE_DIFF('month', act.activation_month, a.activity_month) AS months_since_activation
    FROM activation act
    LEFT JOIN activity a
        ON act.device_id = a.device_id
),

observation_period AS (
    SELECT MAX(activity_month) AS last_activity_month
    FROM activity
)

SELECT
    STRFTIME(ca.activation_month, '%Y-%m') AS activation_month,
    COUNT(DISTINCT ca.device_id) AS cohort_size,
    CASE WHEN ca.activation_month <= op.last_activity_month THEN
        ROUND(100.0 * COUNT(CASE WHEN ca.months_since_activation = 0 THEN ca.device_id END)
            / COUNT(DISTINCT ca.device_id), 2)
    END AS M0_retention,
    CASE WHEN ca.activation_month + INTERVAL '1 month' <= op.last_activity_month THEN
        ROUND(100.0 * COUNT(CASE WHEN ca.months_since_activation = 1 THEN ca.device_id END)
            / COUNT(DISTINCT ca.device_id), 2)
    END AS M1_retention,
    CASE WHEN ca.activation_month + INTERVAL '2 months' <= op.last_activity_month THEN
        ROUND(100.0 * COUNT(CASE WHEN ca.months_since_activation = 2 THEN ca.device_id END)
            / COUNT(DISTINCT ca.device_id), 2)
    END AS M2_retention,
    CASE WHEN ca.activation_month + INTERVAL '3 months' <= op.last_activity_month THEN
        ROUND(100.0 * COUNT(CASE WHEN ca.months_since_activation = 3 THEN ca.device_id END)
            / COUNT(DISTINCT ca.device_id), 2)
    END AS M3_retention
FROM cohort_activity ca
CROSS JOIN observation_period op
GROUP BY ca.activation_month, op.last_activity_month
ORDER BY ca.activation_month DESC;


activation_month,cohort_size,M0_retention,M1_retention,M2_retention,M3_retention
2026-08,6,83.33,66.67,None,None
2026-07,6,100.0,83.33,66.67,None
2026-06,6,100.0,66.67,66.67,66.67


## 5. Partner payouts


In [7]:
%%sql
-- Include devices without a partner in MAU and report their revenue separately with NULL payouts.

WITH activity AS (
    SELECT DISTINCT
        da.device_id,
        STRFTIME(da.activity_date, '%Y-%m') AS activity_month,
        d.partner_id,
        d.country
    FROM device_activity_clean da
    LEFT JOIN devices d
        ON da.device_id = d.device_id
    WHERE da.activity_date >= DATE '2026-07-01'
        AND da.activity_date < DATE '2026-10-01'
),

revenue AS (
    SELECT
        STRFTIME(revenue_month, '%Y-%m') AS revenue_month,
        country,
        SUM(revenue_eur) AS total_revenue
    FROM revenue_by_market
    GROUP BY revenue_month, country
),

revenue_per_device AS (
    SELECT
        a.country,
        a.activity_month,
        MAX(r.total_revenue) / COUNT(DISTINCT a.device_id) AS ARPU
    FROM activity a
    LEFT JOIN revenue r
        ON a.activity_month = r.revenue_month
        AND a.country = r.country
    GROUP BY a.country, a.activity_month
),

monthly_payouts AS (
    SELECT
        rd.activity_month,
        a.partner_id,
        p.partner_name,
        COUNT(DISTINCT a.device_id) AS unique_active_devices,
        SUM(rd.ARPU) AS attributed_revenue,
        SUM(rd.ARPU) * MAX(p.revenue_share_pct) AS partner_payout
    FROM revenue_per_device rd
    LEFT JOIN activity a
        ON rd.country = a.country
        AND rd.activity_month = a.activity_month
    LEFT JOIN partners p
        ON a.partner_id = p.partner_id
    GROUP BY 1, 2, 3
    ORDER BY 1 DESC, 2
)

SELECT
    activity_month,
    partner_name,
    unique_active_devices,
    ROUND(attributed_revenue, 2) AS attributed_revenue,
    ROUND(partner_payout, 2) AS partner_payout,
    ROUND(SUM(partner_payout) OVER (PARTITION BY partner_id ORDER BY activity_month), 2) AS cumulative_payout
FROM monthly_payouts
ORDER BY 2, 1 DESC;


activity_month,partner_name,unique_active_devices,attributed_revenue,partner_payout,cumulative_payout
2026-09,Retailer Norte,5,2751.27,412.69,1285.18
2026-08,Retailer Norte,6,3051.35,457.7,872.49
2026-07,Retailer Norte,5,2765.23,414.78,414.78
2026-09,Retailer Online,5,2354.99,471.0,1112.49
2026-08,Retailer Online,5,1692.19,338.44,641.5
2026-07,Retailer Online,3,1515.3,303.06,303.06
2026-09,Retailer Sur,1,369.37,36.94,123.03
2026-08,Retailer Sur,2,450.05,45.0,86.09
2026-07,Retailer Sur,1,410.85,41.08,41.08
2026-09,None,1,369.37,None,None


## 6. Longest activity streak


In [8]:
%%sql
-- Return the latest streak if lengths are equal.
-- Return a length of 0 and NULL dates for devices without activity.

WITH activity_dates AS (
    SELECT DISTINCT
        device_id,
        activity_date
    FROM device_activity_clean
),

streak_groups AS (
    SELECT
        device_id,
        activity_date,
        activity_date - CAST(ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY activity_date) AS INTEGER) AS streak
    FROM activity_dates
),

streak_ranked AS (
    SELECT
        device_id,
        streak,
        MIN(activity_date) AS min_activity_date,
        MAX(activity_date) AS max_activity_date,
        COUNT(*) AS streak_length,
        ROW_NUMBER() OVER (PARTITION BY device_id ORDER BY COUNT(*) DESC, MIN(activity_date) DESC) AS streak_rank
    FROM streak_groups
    GROUP BY 1, 2
)

SELECT
    d.device_id,
    sr.min_activity_date,
    sr.max_activity_date,
    COALESCE(sr.streak_length, 0) AS streak_length
FROM devices d
LEFT JOIN streak_ranked sr
    ON d.device_id = sr.device_id
    AND sr.streak_rank = 1
ORDER BY 1

device_id,min_activity_date,max_activity_date,streak_length
D001,2026-08-22,2026-08-26,5
D002,2026-09-16,2026-09-18,3
D003,2026-09-10,2026-09-14,5
D004,2026-07-14,2026-07-19,6
D005,2026-07-15,2026-07-21,7
D006,2026-09-05,2026-09-14,10
D007,2026-09-08,2026-09-12,5
D008,2026-09-15,2026-09-24,10
D009,2026-08-19,2026-08-21,3
D010,2026-09-13,2026-09-19,7
